##### 1. Experiment with hyperparameter tuning for a GPT model and document the effects on model performance.
● Choose a set of hyperparameters to tune (e.g., learning rate, batch size, numberof epochs).

● Conduct experiments to observe the impact of different hyperparameter values on model performance.

● Summarize the results in a detailed report (1000-1200 words), including plots or tables to visualize performance changes.
##### 2. Compare and contrast different variants of GPT models in a written report, including their applications and capabilities.
● Research different variants of GPT models (e.g., GPT-2, GPT-3, GPT-Neo, GPT-J).

● Write a comprehensive report (1500-2000 words) comparing their architectures, training data, and performance.

● Discuss the specific applications and capabilities of each variant, providing realworld examples where possible.

In [5]:
import torch 
from transformers import (GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling,)
from datasets import load_dataset 
import math 
import matplotlib.pyplot as plt 
import pandas as pd

In [6]:
# checking gpu availabilty 
device = "cuda" if torch.cuda.is_available() else "cpu" 
print (f"Using {device} device")

Using cuda device


##### Loading the dataset and preparing the dataset

In [7]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
print("Dataset structure:")
print(dataset)
print(f"\nTraining examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['validation'])}")
print(f"\nSample text:")
print(dataset["train"][100]["text"][:300])

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Dataset structure:
DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

Training examples: 36718
Validation examples: 3760

Sample text:
 96 ammunition packing boxes 



##### tokenizing the dataset

In [9]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# gpt 2 doesnt have padding toekn by default so i set one 
tokenizer.pad_token = tokenizer.eos_token 
BLOCK_SIZE = 128

def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=BLOCK_SIZE)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dataset = tokenized_dataset.filter(lambda x: any(t != tokenizer.pad_token_id for t in x["input_ids"]))

print(f"Tokenized training examples: {len(tokenized_dataset['train'])}")
print(f"Tokenized validation examples: {len(tokenized_dataset['validation'])}")
    

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3760 [00:00<?, ? examples/s]

Tokenized training examples: 23767
Tokenized validation examples: 2461


##### creating a training function

In [10]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False,)


##### Training function (i will be running five experiments so i set a training function so i wont need to repeat the function anytime im trying other parameters)

In [11]:
def run_experiment(experiment_name, learning_rate, batch_size, num_epochs):

    print(f"\n{'='*60}")
    print(f"Running: {experiment_name}")
    print(f"  Learning Rate: {learning_rate}")
    print(f"  Batch Size:    {batch_size}")
    print(f"  Epochs:        {num_epochs}")
    print(f"{'='*60}\n")
    
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    model.config.pad_token_id = tokenizer.pad_token_id
    
    training_args = TrainingArguments(
        output_dir=f"./results/{experiment_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        weight_decay=0.01,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=data_collator,
    )
    
    trainer.train()
    
    eval_results = trainer.evaluate()
    eval_loss = eval_results["eval_loss"]
    perplexity = math.exp(eval_loss)
    
    print(f"\n--- Results for {experiment_name} ---")
    print(f"  Eval Loss:   {eval_loss:.4f}")
    print(f"  Perplexity:  {perplexity:.2f}")
    
    return {
        "experiment": experiment_name,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "epochs": num_epochs,
        "eval_loss": round(eval_loss, 4),
        "perplexity": round(perplexity, 2),
    }

all_results = []

##### Baseline- the starting settings



In [12]:
baseline = run_experiment("Baseline", learning_rate=5e-5, batch_size=8, num_epochs=3)
all_results.append(baseline)



Running: Baseline
  Learning Rate: 5e-05
  Batch Size:    8
  Epochs:        3



config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.338500,3.346378
2,3.088114,3.344941
3,2.988951,3.352178



--- Results for Baseline ---
  Eval Loss:   3.3522
  Perplexity:  28.56


##### Experiment 1- Low learning rate

In [13]:
exp_a1 = run_experiment("Exp_A1_Low_LR", learning_rate=1e-5, batch_size=8, num_epochs=3)
all_results.append(exp_a1)


Running: Exp_A1_Low_LR
  Learning Rate: 1e-05
  Batch Size:    8
  Epochs:        3



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,3.453148,3.380861
2,3.332605,3.359735
3,3.320699,3.354295



--- Results for Exp_A1_Low_LR ---
  Eval Loss:   3.3543
  Perplexity:  28.63


##### Experiment 2 - high learning rate

In [14]:
exp_a2 = run_experiment("Exp_A2_High_LR", learning_rate=5e-4, batch_size=8, num_epochs=3)
all_results.append(exp_a2)


Running: Exp_A2_High_LR
  Learning Rate: 0.0005
  Batch Size:    8
  Epochs:        3



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,3.592124,3.797683
2,2.862806,3.790728
3,2.213489,3.979901



--- Results for Exp_A2_High_LR ---
  Eval Loss:   3.9799
  Perplexity:  53.51
